# Churn Prediction

## Objective

The objective of this analysis is to predict whether a customer is likely to churn based on their historical purchasing behavior.

For this project, a customer is considered **churned** if they make no purchase during the 90-day period after the selected cutoff date.

Customer purchase behavior before the cutoff date will be used to create features for the prediction model.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/online_retail_cleaned.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["TotalAmount"] = df["Quantity"] * df["Price"]

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalAmount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [2]:
print("Start date:", df["InvoiceDate"].min())
print("End date:", df["InvoiceDate"].max())

Start date: 2009-12-01 07:45:00
End date: 2011-12-09 12:50:00


In [3]:
cutoff_date = pd.Timestamp("2011-09-10")

print("Cutoff date:", cutoff_date)
print("Dataset end date:", df["InvoiceDate"].max())

Cutoff date: 2011-09-10 00:00:00
Dataset end date: 2011-12-09 12:50:00


In [4]:
feature_df = df[df["InvoiceDate"] < cutoff_date].copy()

print("Feature data shape:", feature_df.shape)
print("Feature period:", feature_df["InvoiceDate"].min(), "to", feature_df["InvoiceDate"].max())

Feature data shape: (620572, 9)
Feature period: 2009-12-01 07:45:00 to 2011-09-09 15:53:00


In [5]:
future_df = df[
    (df["InvoiceDate"] >= cutoff_date) &
    (df["InvoiceDate"] <= df["InvoiceDate"].max())
].copy()

print("Future data shape:", future_df.shape)
print("Future period:", future_df["InvoiceDate"].min(), "to", future_df["InvoiceDate"].max())

Future data shape: (158853, 9)
Future period: 2011-09-11 10:35:00 to 2011-12-09 12:50:00


In [6]:
future_customers = future_df["Customer ID"].unique()

print("Customers with future purchases:", len(future_customers))

Customers with future purchases: 2889


In [7]:
feature_customers = feature_df["Customer ID"].unique()

print("Customers in feature period:", len(feature_customers))

Customers in feature period: 5281


In [8]:
customer_labels = pd.DataFrame({
    "Customer ID": feature_customers
})

customer_labels["Churn"] = (
    ~customer_labels["Customer ID"].isin(future_customers)
).astype(int)

customer_labels["Churn"].value_counts()

Churn
1    2989
0    2292
Name: count, dtype: int64

In [9]:
customer_features = feature_df.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (cutoff_date - x.max()).days)
).reset_index()

customer_features.head()

,Customer ID,Recency
0,12346.0,234
1,12347.0,38
2,12348.0,157
3,12349.0,316
4,12350.0,219


In [10]:
frequency = feature_df.groupby("Customer ID")["Invoice"].nunique().reset_index()

frequency.rename(columns={"Invoice": "Frequency"}, inplace=True)

frequency.head()

,Customer ID,Frequency
0,12346.0,12
1,12347.0,6
2,12348.0,4
3,12349.0,3
4,12350.0,1


In [11]:
monetary = feature_df.groupby("Customer ID")["TotalAmount"].sum().reset_index()

monetary.rename(columns={"TotalAmount": "Monetary"}, inplace=True)

monetary.head()

,Customer ID,Monetary
0,12346.0,77556.46
1,12347.0,3402.39
2,12348.0,1709.40
3,12349.0,2671.14
4,12350.0,334.40


In [12]:
customer_features = customer_features.merge(
    frequency,
    on="Customer ID"
).merge(
    monetary,
    on="Customer ID"
)

customer_features.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,234,12,77556.46
1,12347.0,38,6,3402.39
2,12348.0,157,4,1709.40
3,12349.0,316,3,2671.14
4,12350.0,219,1,334.40


In [13]:
customer_data = customer_features.merge(
    customer_labels,
    on="Customer ID"
)

customer_data.head()

,Customer ID,Recency,Frequency,Monetary,Churn
0,12346.0,234,12,77556.46,1
1,12347.0,38,6,3402.39,0
2,12348.0,157,4,1709.40,0
3,12349.0,316,3,2671.14,0
4,12350.0,219,1,334.40,1


In [14]:
print("Shape:", customer_data.shape)
print("\nMissing values:")
print(customer_data.isnull().sum())
print("\nData types:")
print(customer_data.dtypes)

Shape: (5281, 5)

Missing values:
Customer ID    0
Recency        0
Frequency      0
Monetary       0
Churn          0
dtype: int64

Data types:
Customer ID    float64
Recency          int64
Frequency        int64
Monetary       float64
Churn            int64
dtype: object


In [15]:
print(customer_data["Churn"].value_counts())
print("\nChurn percentage:")
print(customer_data["Churn"].value_counts(normalize=True) * 100)

Churn
1    2989
0    2292
Name: count, dtype: int64

Churn percentage:
Churn
1    56.599129
0    43.400871
Name: proportion, dtype: float64


In [16]:
X = customer_data[["Recency", "Frequency", "Monetary"]]
y = customer_data["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5281, 3)
y shape: (5281,)


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (4224, 3)
Testing data: (1057, 3)


In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following a

In [20]:
y_pred = model.predict(X_test)

print(y_pred[:20])

[1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 0 0 1 1 1]


In [21]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.7360454115421002


In [22]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)


[[312 147]
 [132 466]]


In [23]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.68      0.69       459
           1       0.76      0.78      0.77       598

    accuracy                           0.74      1057
   macro avg       0.73      0.73      0.73      1057
weighted avg       0.74      0.74      0.74      1057



In [24]:
results = X_test.copy()

results["Actual_Churn"] = y_test
results["Predicted_Churn"] = y_pred

results.head(10)

,Recency,Frequency,Monetary,Actual_Churn,Predicted_Churn
2984,465,2,420.25,1,1
5192,170,2,529.40,1,1
3770,275,3,329.48,0,1
5027,271,1,157.92,1,1
5176,512,1,58.50,0,1
1735,151,2,516.45,1,1
1514,175,1,152.55,1,1
3295,340,1,89.65,1,1
2924,323,2,443.16,0,1
2462,197,1,295.15,0,1


In [25]:
y_prob = model.predict_proba(X_test)[:, 1]

print(y_prob[:20])

[0.86748416 0.61351    0.7045335  0.74057877 0.90083416 0.59170469
 0.64297082 0.79911115 0.76802133 0.66654448 0.77535407 0.46300451
 0.92162385 0.8022659  0.90655051 0.39472821 0.46371418 0.73881819
 0.80105254 0.76342767]


In [26]:
results["Churn_Probability"] = y_prob

results.head(10)

,Recency,Frequency,Monetary,Actual_Churn,Predicted_Churn,Churn_Probability
2984,465,2,420.25,1,1,0.867484
5192,170,2,529.40,1,1,0.613510
3770,275,3,329.48,0,1,0.704534
5027,271,1,157.92,1,1,0.740579
5176,512,1,58.50,0,1,0.900834
1735,151,2,516.45,1,1,0.591705
1514,175,1,152.55,1,1,0.642971
3295,340,1,89.65,1,1,0.799111
2924,323,2,443.16,0,1,0.768021
2462,197,1,295.15,0,1,0.666544


In [27]:
X.describe()

,Recency,Frequency,Monetary
count,5281.000000,5281.000000,5281.000000
mean,206.814240,5.745692,2637.863286
std,175.029112,11.464582,12225.544955
min,0.000000,1.000000,2.900000
25%,49.000000,1.000000,318.240000
50%,163.000000,3.000000,783.580000
75%,326.000000,6.000000,2078.540000
max,647.000000,309.000000,456780.490000


In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [29]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [30]:
scaled_model = LogisticRegression(random_state=42)

scaled_model.fit(X_train_scaled, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following a

In [31]:
y_pred_scaled = scaled_model.predict(X_test_scaled)

print("Scaled Model Accuracy:",
      accuracy_score(y_test, y_pred_scaled))

Scaled Model Accuracy: 0.7360454115421002


In [32]:
y_pred_scaled = scaled_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred_scaled))

              precision    recall  f1-score   support

           0       0.70      0.68      0.69       459
           1       0.76      0.78      0.77       598

    accuracy                           0.74      1057
   macro avg       0.73      0.73      0.73      1057
weighted avg       0.74      0.74      0.74      1057



In [33]:
cm_scaled = confusion_matrix(y_test, y_pred_scaled)

print(cm_scaled)

[[312 147]
 [132 466]]


In [34]:
scaled_model.coef_

array([[ 0.84012928, -1.00108768, -0.10227018]])

In [35]:
scaled_model.intercept_

array([0.26700947])

In [36]:
results = X_test.copy()

results["Actual_Churn"] = y_test
results["Predicted_Churn"] = y_pred_scaled

results["Churn_Probability"] = scaled_model.predict_proba(X_test_scaled)[:, 1]

results.head(10)

,Recency,Frequency,Monetary,Actual_Churn,Predicted_Churn,Churn_Probability
2984,465,2,420.25,1,1,0.867168
5192,170,2,529.40,1,1,0.613163
3770,275,3,329.48,0,1,0.704342
5027,271,1,157.92,1,1,0.740071
5176,512,1,58.50,0,1,0.900504
1735,151,2,516.45,1,1,0.591374
1514,175,1,152.55,1,1,0.642465
3295,340,1,89.65,1,1,0.798642
2924,323,2,443.16,0,1,0.767645
2462,197,1,295.15,0,1,0.666016


In [37]:
results.to_csv(
    "../data/processed/churn_predictions.csv",
    index=False
)

print("Churn predictions saved successfully.")

Churn predictions saved successfully.


In [38]:
import os

print(os.path.exists("../data/processed/churn_predictions.csv"))
print(pd.read_csv("../data/processed/churn_predictions.csv").shape)

True
(1057, 6)


## Churn Model Summary

A Logistic Regression model was developed to predict customer churn using historical purchasing behavior.

### Churn Definition

For this project, a customer is considered churned if they made no purchase during the 90-day period after the selected cutoff date.

### Features Used

The model uses three customer-level features calculated before the cutoff date:

* **Recency:** Days since the customer's most recent purchase.
* **Frequency:** Number of unique invoices.
* **Monetary:** Total amount spent.

The features were standardized using `StandardScaler` before training the Logistic Regression model.

### Model Evaluation

The model achieved:

* **Accuracy:** 73.6%
* **Churn Precision:** 76%
* **Churn Recall:** 78%
* **Churn F1-score:** 77%

The confusion matrix was:

```text
[[312, 147],
 [132, 466]]
```

The model's coefficients indicate that higher Recency is associated with higher predicted churn probability, while higher Frequency and Monetary values are associated with lower predicted churn probability.

The test-set predictions and churn probabilities were saved to:

`data/processed/churn_predictions.csv`
